In [43]:
import pandas as pd

df = pd.read_csv("../data/raw_student_performance.csv")


print(
    df.loc[df["age"] < 16, "age"]
      .value_counts()
      .sort_index()
)

age
3.0      26
10.0      3
11.0      6
12.0     23
13.0    102
14.0    346
15.0    925
Name: count, dtype: int64


In [258]:
import pandas as pd

df = pd.read_csv("../data/raw_student_performance.csv")


print(df["age"].describe())

count    100500.000000
mean         20.998000
std           2.533943
min           3.000000
25%          19.000000
50%          21.000000
75%          23.000000
max          32.000000
Name: age, dtype: float64


In [356]:
duplicate_ids = df[
    df["student_id"].duplicated(keep=False)
].sort_values("student_id")

print(duplicate_ids)

       student_id   age         department  level  study_hours  \
946        100087  22.0            Physics    100     2.122790   
57539      100087  22.0            Physics    100     2.122790   
440        100149  22.0  Civil Engineering    200    14.969781   
9176       100149  22.0  Civil Engineering    200    14.969781   
74546      100226  21.0  Civil Engineering    400     3.760647   
...           ...   ...                ...    ...          ...   
92993      199476  17.0     Cyber Security    200     5.813276   
19564      199836  20.0     Cyber Security    300     0.822429   
38866      199836  20.0     Cyber Security    300     0.822429   
99161      199906  23.0  Civil Engineering    400     5.357922   
71115      199906  23.0  Civil Engineering    400     5.357922   

       attendance_rate  previous_gpa  assignment_average  quiz_average  \
946         100.000000      2.754504           72.973568     79.776219   
57539       100.000000      2.754504           72.973568   

In [357]:
duplicate_mask = df.duplicated(keep=False)
print("Duplicate rows:", duplicate_mask.sum())
missing_mask = df.isnull().any(axis=1)
print("Rows with missing values:", missing_mask.sum())
invalid_mask = (
    (df["study_hours"] < 0)
    | (df["attendance_rate"] > 100)
    | (df["previous_gpa"] > 5)
    | (df["age"] < 16)
)
print("Rows with invalid values:", invalid_mask.sum())

print("\nDuplicate + missing:",
      (duplicate_mask & missing_mask).sum())

print("Duplicate + invalid:",
      (duplicate_mask & invalid_mask).sum())

print("Missing + invalid:",
      (missing_mask & invalid_mask).sum())

Duplicate rows: 1000
Rows with missing values: 9645
Rows with invalid values: 1506

Duplicate + missing: 106
Duplicate + invalid: 20
Missing + invalid: 154


In [ ]:
duplicated_row=df.duplicated(keep=False)
print("Duplicated_rows: ",duplicated_row.sum())
missing_values=df.isnull().any(axis=1)
print("Missing_values_count: ",missing_values.sum())
invalid_values=(
    (df["age"] < 16)
    | (df["previous_gpa"] > 5)
    | (df["attendance_rate"] > 100)
    | (df["study_hours"]<0)
)                
print("invalid_data: ",invalid_values.sum())

print("\nDuplicated + missing Data: ",(duplicated_row + missing_values).sum())

print("Missing + Invalid Data: ", (missing_values + invalid_values).sum())

print("Invalid + Duplicated: ", (invalid_values + duplicated_row).sum())

Duplicated_rows:  1000
Missing_values_count:  9645
invalid_data:  1506


In [391]:
clean_df = df.drop_duplicates()

print("Before:", len(df))
print("After:", len(clean_df))
print("Removed:", len(df) - len(clean_df))

print("Remaining duplicates:",
      clean_df.duplicated().sum())

print("Unique students:",
      clean_df["student_id"].nunique())

print(clean_df.shape)

Before: 100500
After: 100000
Removed: 500
Remaining duplicates: 0
Unique students: 100000
(100000, 16)


In [392]:

# Count violations after removing exact duplicates

checks = {
    "Negative study hours": clean_df["study_hours"] < 0,

    "Attendance outside 0–100": (
        (clean_df["attendance_rate"] < 0)
        | (clean_df["attendance_rate"] > 100)
    ),

    "GPA outside 0–5": (
        (clean_df["previous_gpa"] < 0)
        | (clean_df["previous_gpa"] > 5)
    ),

    "Age below 16": clean_df["age"] < 16,
}

for issue, mask in checks.items():
    print(f"{issue}: {mask.sum():,}")


Negative study hours: 25
Attendance outside 0–100: 25
GPA outside 0–5: 25
Age below 16: 1,422


In [393]:

import numpy as np
import pandas as pd
df = pd.read_csv("../data/raw_student_performance.csv")


# Start from the deduplicated dataset
clean_df = df.drop_duplicates().copy()

# Record the number of invalid values before correction
invalid_counts_before = {
    "study_hours": (clean_df["study_hours"] < 0).sum(),
    "attendance_rate": (
        (clean_df["attendance_rate"] < 0)
        | (clean_df["attendance_rate"] > 100)
    ).sum(),
    "previous_gpa": (
        (clean_df["previous_gpa"] < 0)
        | (clean_df["previous_gpa"] > 5)
    ).sum(),
}

print("Invalid values before cleaning:")
print(invalid_counts_before)

# Replace invalid values with NaN
clean_df.loc[
    clean_df["study_hours"] < 0,
    "study_hours"
] = np.nan

clean_df.loc[
    ~clean_df["attendance_rate"].between(0, 100),
    "attendance_rate"
] = np.nan

clean_df.loc[
    ~clean_df["previous_gpa"].between(0, 5),
    "previous_gpa"
] = np.nan

print("\nMissing values after correction:")
print(
    clean_df[
        ["study_hours", "attendance_rate", "previous_gpa"]
    ].isna().sum()
)


Invalid values before cleaning:
{'study_hours': np.int64(25), 'attendance_rate': np.int64(25), 'previous_gpa': np.int64(25)}

Missing values after correction:
study_hours        2025
attendance_rate    2025
previous_gpa       2025
dtype: int64


In [394]:

print("Negative study hours remaining:",
      (clean_df["study_hours"] < 0).sum())

print("Attendance outside 0–100 remaining:",
      (~clean_df["attendance_rate"].between(0, 100)).sum())

print("GPA outside 0–5 remaining:",
      (~clean_df["previous_gpa"].between(0, 5)).sum())

print("\nDataset shape:", clean_df.shape)


Negative study hours remaining: 0
Attendance outside 0–100 remaining: 2025
GPA outside 0–5 remaining: 2025

Dataset shape: (100000, 16)


In [395]:

# Negative study hours, excluding missing values
print(
    "Negative study hours remaining:",
    (clean_df["study_hours"] < 0).sum()
)

# Attendance: present values outside 0–100
attendance_invalid = (
    clean_df["attendance_rate"].notna()
    & ~clean_df["attendance_rate"].between(0, 100)
)

print(
    "Attendance outside 0–100 remaining:",
    attendance_invalid.sum()
)

# GPA: present values outside 0–5
gpa_invalid = (
    clean_df["previous_gpa"].notna()
    & ~clean_df["previous_gpa"].between(0, 5)
)

print(
    "GPA outside 0–5 remaining:",
    gpa_invalid.sum()
)

print("\nDataset shape:", clean_df.shape)


Negative study hours remaining: 0
Attendance outside 0–100 remaining: 0
GPA outside 0–5 remaining: 0

Dataset shape: (100000, 16)


In [396]:

# Count missing values per column
missing_summary = pd.DataFrame({
    "missing_count": clean_df.isna().sum(),
    "missing_percentage": (
        clean_df.isna().mean() * 100
    ).round(2),
})

missing_summary = missing_summary.sort_values(
    "missing_count",
    ascending=False
)

print(missing_summary)


                       missing_count  missing_percentage
attendance_rate                 2025                2.02
previous_gpa                    2025                2.02
study_hours                     2025                2.02
assignment_average              2000                2.00
quiz_average                    2000                2.00
level                              0                0.00
age                                0                0.00
department                         0                0.00
student_id                         0                0.00
midterm_score                      0                0.00
internet_access                    0                0.00
library_visits                     0                0.00
sleep_hours                        0                0.00
previous_failures                  0                0.00
extracurricular_hours              0                0.00
final_score                        0                0.00


In [397]:

columns_to_inspect = [
    "study_hours",
    "attendance_rate",
    "previous_gpa",
    "assignment_average",
    "quiz_average",
]

print(clean_df[columns_to_inspect].describe().T)

print("\nSkewness:")
print(clean_df[columns_to_inspect].skew().round(3))


                      count       mean        std        min        25%  \
study_hours         97975.0   4.505636   2.576160   0.063578   2.597116   
attendance_rate     97975.0  77.809912  11.654263  27.820918  69.897628   
previous_gpa        97975.0   2.800662   0.701343   0.000000   2.328913   
assignment_average  98000.0  67.404265  11.448648  17.887189  59.664127   
quiz_average        98000.0  65.143494  13.517399   9.030365  55.998146   

                          50%        75%    max  
study_hours          4.027193   5.901038   15.0  
attendance_rate     78.000175  86.098864  100.0  
previous_gpa         2.802050   3.271763    5.0  
assignment_average  67.397955  75.147675  100.0  
quiz_average        65.122183  74.326789  100.0  

Skewness:
study_hours           1.055
attendance_rate      -0.174
previous_gpa         -0.003
assignment_average   -0.006
quiz_average         -0.028
dtype: float64


In [398]:

columns_to_impute = [
    "study_hours",
    "attendance_rate",
    "previous_gpa",
    "assignment_average",
    "quiz_average",
]

# Save the missing-value counts before imputation
missing_before = clean_df[columns_to_impute].isna().sum()

print("Missing values before imputation:")
print(missing_before)

# Calculate medians from the currently available values
medians = clean_df[columns_to_impute].median()

# Fill missing values
clean_df[columns_to_impute] = (
    clean_df[columns_to_impute].fillna(medians)
)

print("\nMissing values after imputation:")
print(clean_df[columns_to_impute].isna().sum())

print("\nDataset shape:", clean_df.shape)


Missing values before imputation:
study_hours           2025
attendance_rate       2025
previous_gpa          2025
assignment_average    2000
quiz_average          2000
dtype: int64

Missing values after imputation:
study_hours           0
attendance_rate       0
previous_gpa          0
assignment_average    0
quiz_average          0
dtype: int64

Dataset shape: (100000, 16)


In [399]:

# 1. Missing values across every column
print("Total missing values:", clean_df.isna().sum().sum())

# 2. Duplicate rows
print("Duplicate rows:", clean_df.duplicated().sum())

# 3. Unique student IDs
print("Unique student IDs:", clean_df["student_id"].nunique())

# 4. Domain constraints
print("Negative study hours:",
      (clean_df["study_hours"] < 0).sum())

print("Attendance outside 0–100:",
      (~clean_df["attendance_rate"].between(0, 100)).sum())

print("GPA outside 0–5:",
      (~clean_df["previous_gpa"].between(0, 5)).sum())

# 5. Dataset dimensions
print("Shape:", clean_df.shape)


Total missing values: 0
Duplicate rows: 0
Unique student IDs: 100000
Negative study hours: 0
Attendance outside 0–100: 0
GPA outside 0–5: 0
Shape: (100000, 16)
